# S02 Tutorial: From Recursion to Algorithm Complexity

**Module:** M12 — Data Structures and Algorithms  
**Level:** Beginner-friendly, step by step  
**Language:** Python 3 (Google Colab compatible)

## What you will learn
By the end of this tutorial, you will be able to:
1. Describe how a recursive function calls itself and eventually stops.
2. Translate a recursive algorithm into a recurrence relation.
3. Solve suitable recurrences using the Master Theorem.
4. Implement binary search and explain its time and memory costs.
5. Use Big O to compare algorithms.

### How to use this notebook
Run the code cells from top to bottom. Before running a cell marked **Your turn**, try to complete it yourself. Read the explanation after each experiment. No external packages are required.

## 0. Warm-up: what does an algorithm cost?
An **algorithm** is a sequence of steps that solves a problem. The input size, usually written as `n`, tells us how much data the algorithm receives.

For example, if we search a list containing 10 numbers, then `n = 10`. If the list contains 1,000 numbers, then `n = 1,000`.

**Time complexity** describes how the amount of work grows as `n` grows. **Space complexity** describes how the extra memory grows. We usually study growth rather than seconds, because seconds depend on the computer and its current workload.

In [ ]:
numbers = [4, 8, 15, 16, 23, 42]
n = len(numbers)
print("Input:", numbers)
print("Input size n:", n)

## 1. Recursion: the smallest building blocks
A function is **recursive** when it calls itself to solve a smaller version of the same problem.

Every safe recursive function needs:
- A **base case**: the simple situation that can be answered immediately. It stops the recursion.
- A **recursive case**: the function calls itself with a smaller or simpler input.
- **Progress**: each call must move towards the base case.

### Example: countdown
We want to print 3, 2, 1 and then stop. The base case is `n == 0`; otherwise we print `n` and call the function with `n - 1`.

In [ ]:
def countdown(n):
    if n == 0:                 # base case: stop
        print("Done!")
        return
    print(n)
    countdown(n - 1)           # recursive case: smaller input

countdown(3)

### What happens during the calls?
The calls are `countdown(3)`, `countdown(2)`, `countdown(1)`, and `countdown(0)`. At zero, the base case runs. Then the calls finish in reverse order.

The **call stack** is the memory area that keeps track of unfinished function calls. Each call has a stack frame. When a call returns, its frame is removed.

### Example: factorial
The factorial of a non-negative integer is the product of all positive integers up to that number. For example, `4! = 4 × 3 × 2 × 1 = 24`. The definition is `0! = 1` and `n! = n × (n-1)!` for `n > 0`.

In [ ]:
def factorial(n):
    if n < 0:
        raise ValueError("n must be non-negative")
    if n == 0:                 # base case
        return 1
    return n * factorial(n - 1)

print("factorial(4) =", factorial(4))

### Your turn 1
1. What is the base case in `factorial`?
2. What smaller input is passed to the next call?
3. Predict `factorial(3)` before running the next cell.

A useful warning: a recursive function without a reachable base case, or without progress towards it, may keep calling itself until Python raises a `RecursionError`.

In [ ]:
# Try changing 3 to another non-negative integer.
factorial(3)

## 2. Recurrence relations: describing the work with mathematics
A **recurrence relation** describes the cost of a recursive algorithm in terms of the cost of smaller input(s).

We write `T(n)` for the running time on an input of size `n`. A recurrence normally has:
- the cost of the recursive call or calls;
- the work done outside those calls;
- a base condition for a small input.

### Example A: one smaller problem
In `factorial(n)`, there is one call on `n - 1` and a constant amount of extra work (a comparison and multiplication). A simplified model is:

`T(n) = T(n - 1) + Θ(1)`, with `T(0) = Θ(1)`.

Expanding a few steps gives `T(n) = T(n-2) + Θ(1) + Θ(1)`, and eventually there are about `n` constant-cost steps. Therefore `T(n) = Θ(n)`.

### Example B: divide into two halves
Imagine an algorithm that solves two problems, each half the original size, and then scans all `n` items to combine the answers:

`T(n) = 2T(n/2) + Θ(n)`.

For `n = 8`, the first level has one problem of size 8. The next level has two problems of size 4. The following level has four problems of size 2, then eight base problems of size 1.

At each non-leaf level, the total combining work is proportional to `n`: there are more subproblems, but each is smaller. There are about `log₂(n)` levels, so the total is `Θ(n log n)`.

In [ ]:
# Exact level counts for a problem of size 8.
# Each level has twice as many subproblems, each half as large.
for level, (count, subproblem_size) in enumerate([(1, 8), (2, 4), (4, 2), (8, 1)]):
    print(f"Level {level}: {count} subproblem(s) of size {subproblem_size}")

### Your turn 2: write the recurrence
An algorithm makes **one** recursive call on an input of size `n/2` and then performs a constant amount of extra work.

Fill in the blanks:

`T(n) = ________ + ________`

How many times can you halve `n` before it reaches 1? That number of halvings is approximately `log₂(n)`. This is why the recurrence has logarithmic growth.

## 3. Master Theorem: solving a common family of recurrences
The **Master Theorem** is a shortcut for solving recurrences of this form:

`T(n) = a T(n / b) + Θ(n^d)`

where `a ≥ 1` is the number of recursive subproblems, `b > 1` is the factor by which each subproblem shrinks, and `n^d` describes the work outside the recursive calls (with `d ≥ 0`).

First calculate `p = log_b(a)`. Then compare `d` with `p`:

| Comparison | Result (under the standard theorem conditions) |
|---|---|
| `d < p` | `T(n) = Θ(n^p)` |
| `d = p` | `T(n) = Θ(n^p log n)` |
| `d > p` | `T(n) = Θ(n^d)` (requires the regularity condition) |

These are simplified polynomial-work cases. The full theorem has more precise conditions. Do not apply this shortcut automatically to every recurrence: unequal subproblem sizes, unusual `f(n)`, or a failed regularity condition may require another method.

### Worked example: `T(n) = 2T(n/2) + Θ(n)`
Step 1: `a = 2`, because there are two recursive calls.  
Step 2: `b = 2`, because each call receives half the input.  
Step 3: outside work is `Θ(n)`, so `d = 1`.  
Step 4: calculate `p = log₂(2) = 1`.  
Step 5: compare `d = 1` and `p = 1`: they are equal.  
Therefore `T(n) = Θ(n log n)`.

This is the recurrence commonly used to analyse Merge Sort.

In [ ]:
import math

a = 2
b = 2
d = 1
p = math.log(a, b)
print("p = log_b(a) =", p)
if d < p:
    print("Case 1: Theta(n^p)")
elif d == p:
    print("Case 2: Theta(n^p log n)")
else:
    print("Case 3: Theta(n^d), if the regularity condition holds")

### Your turn 3
For `T(n) = 4T(n/2) + Θ(n)`, identify `a`, `b`, and `d`. Calculate `p = log_b(a)`. Is `d` smaller than, equal to, or greater than `p`? Use the table to state the result.

**Important:** the Master Theorem gives an asymptotic result. It does not give the exact number of seconds an implementation will take.

## 4. Binary search: using repeated halving to find a value
Binary search finds a target in a **sorted** sequence. It looks at the middle item and discards the half that cannot contain the target. It repeats until it finds the target or no items remain.

**Precondition:** the input must be sorted in ascending order. Binary search is not valid on an arbitrary unsorted list.

Example list: `[2, 4, 7, 9, 13, 18, 21]`. We want to find `13`.
1. Look at the middle item.
2. If it equals 13, stop.
3. If it is smaller than 13, continue in the right half.
4. If it is larger than 13, continue in the left half.

In [ ]:
def binary_search(values, target):
    """Return an index containing target, or -1 if target is absent.
    values must be sorted in ascending order.
    """
    low = 0
    high = len(values)  # search interval is [low, high)

    while low < high:
        middle = low + (high - low) // 2
        if values[middle] == target:
            return middle
        elif values[middle] < target:
            low = middle + 1
        else:
            high = middle

    return -1

data = [2, 4, 7, 9, 13, 18, 21]
print("13 is at index:", binary_search(data, 13))
print("8 is at index:", binary_search(data, 8))

### Understand the boundaries
Python indexes start at zero. For a list of length 7, valid indexes are 0 through 6. We use the half-open interval `[low, high)`: `low` is included and `high` is excluded. Initially, `low = 0` and `high = len(values)`.

If the middle value is too small, we set `low = middle + 1` because the middle item has already been checked. If it is too large, we set `high = middle` because the middle item and everything after it can be excluded.

The interval gets smaller on every iteration. This guarantees progress and prevents an infinite loop.

In [ ]:
# Trace binary search one step at a time.
def trace_binary_search(values, target):
    low, high = 0, len(values)
    step = 1
    while low < high:
        middle = low + (high - low) // 2
        print(f"Step {step}: low={low}, high={high}, middle={middle}, value={values[middle]}")
        if values[middle] == target:
            print("Found at index", middle)
            return middle
        if values[middle] < target:
            low = middle + 1
        else:
            high = middle
        step += 1
    print("Not found")
    return -1

trace_binary_search([2, 4, 7, 9, 13, 18, 21], 13)

### Recursive binary search
The same idea can be written recursively. Each call searches a smaller interval. When the interval is empty (`low >= high`), the target is absent: this is the base case.

Its recurrence is `T(n) = T(n/2) + Θ(1)`: one recursive call on roughly half the items, plus constant work to calculate and compare the middle item. By the Master Theorem, the time complexity is `Θ(log n)`.

In [ ]:
def binary_search_recursive(values, target, low=0, high=None):
    if high is None:
        high = len(values)
    if low >= high:                 # base case: empty interval
        return -1

    middle = low + (high - low) // 2
    if values[middle] == target:
        return middle
    if values[middle] < target:
        return binary_search_recursive(values, target, middle + 1, high)
    return binary_search_recursive(values, target, low, middle)

data = [2, 4, 7, 9, 13, 18, 21]
print(binary_search_recursive(data, 13))
print(binary_search_recursive(data, 8))

### Your turn 4: test the edge cases
Run the following tests. Add at least two more cases of your own. Check both implementations.

Think about: an empty list, one item, the first item, the last item, a missing item, and repeated values. With repeated values, these implementations may return **any** matching index; they do not promise the first or last occurrence.

In [ ]:
test_cases = [
    ([], 5),
    ([7], 7),
    ([7], 3),
    ([1, 3, 5, 7, 9], 1),
    ([1, 3, 5, 7, 9], 9),
    ([1, 3, 5, 7, 9], 6),
    ([2, 2, 2, 5, 5, 9], 5),
]

for values, target in test_cases:
    iterative = binary_search(values, target)
    recursive = binary_search_recursive(values, target)
    print(f"values={values}, target={target}: iterative={iterative}, recursive={recursive}")
    for index in (iterative, recursive):
        assert index == -1 or values[index] == target
print("All checks passed.")

## 5. Complexity analysis: putting the ideas together
Big O describes an upper-bound growth rate. In introductory comparisons, we often use it to communicate how an algorithm scales as input size increases. A tight bound is often written with `Θ`.

| Algorithm | Time complexity | Extra space (typical implementation) |
|---|---:|---:|
| Linear search | `O(n)` | `O(1)` |
| Iterative binary search | `O(log n)` | `O(1)` |
| Recursive binary search | `O(log n)` | `O(log n)` call stack |
| Merge Sort | `O(n log n)` | commonly `O(n)` auxiliary array space |

The input list itself is not counted as auxiliary space here. Recursive binary search uses extra stack frames; iterative binary search uses a fixed number of variables.

### Why does binary search take logarithmic time?
Each step halves the remaining search interval. After one step, about `n/2` items remain; after two, `n/4`; after `k` steps, about `n/2^k`. We stop when this is at most 1. Solving `n/2^k ≤ 1` gives `k ≥ log₂(n)`. Therefore the number of steps grows as `O(log n)`.

In [ ]:
def linear_search(values, target):
    for index, value in enumerate(values):
        if value == target:
            return index
    return -1

# Count comparisons, rather than relying on noisy wall-clock timings.
def linear_search_count(values, target):
    comparisons = 0
    for index, value in enumerate(values):
        comparisons += 1
        if value == target:
            return index, comparisons
    return -1, comparisons

def binary_search_count(values, target):
    low, high = 0, len(values)
    comparisons = 0
    while low < high:
        middle = low + (high - low) // 2
        comparisons += 1
        if values[middle] == target:
            return middle, comparisons
        if values[middle] < target:
            low = middle + 1
        else:
            high = middle
    return -1, comparisons

for size in [10, 100, 1_000, 10_000]:
    values = list(range(size))
    target = size - 1  # worst case for linear search; near the end for binary search
    _, linear_checks = linear_search_count(values, target)
    _, binary_checks = binary_search_count(values, target)
    print(f"n={size:>5}: linear comparisons={linear_checks:>5}, binary comparisons={binary_checks:>2}")

### Why comparison counts are useful
Wall-clock timing can vary because of the computer, background processes, interpreter overhead and small input sizes. Counting comparisons gives a simple, repeatable way to observe algorithmic growth. It is not a complete performance benchmark, but it supports the theoretical analysis.

Binary search is not always the best choice: the data must be sorted, and maintaining sorted order can be expensive when data changes frequently. For a short unsorted list or a single lookup, linear search may be perfectly reasonable.

## 6. Final integrated exercise
Complete these steps using the code and explanations above. Write your answers in a Markdown cell below.

1. Implement iterative and recursive binary search for a sorted list of your choice.
2. Test an empty list, a one-item list, a successful search and an unsuccessful search.
3. Explain why the recursive version terminates. Identify its base case and its progress towards that case.
4. Write the recurrence for recursive binary search and explain each term: `T(n) = T(n/2) + Θ(1)`.
5. Apply the Master Theorem: identify `a`, `b`, and `d`, calculate `log_b(a)`, and state the resulting time complexity.
6. State the auxiliary-space complexity of each implementation and explain the difference.
7. Compare binary search with linear search. Explain the sorted-input precondition.

### Reflection questions
- What is the difference between a recursive function and a recurrence relation?
- Why can we not use binary search on an unsorted list?
- Why does recursive binary search use more auxiliary space than iterative binary search?
- When is the basic Master Theorem not appropriate?

## 7. Summary: the four ideas are connected
- **Recursion** is a programming technique: a function calls itself on a smaller problem.
- **Recurrence relations** are mathematical models of the work performed by recursive algorithms.
- **Master Theorem** solves a useful family of recurrence relations, when its conditions are met.
- **Binary search** is a practical divide-and-conquer example: it repeatedly halves a sorted search interval.
- **Complexity analysis** uses these models to explain how time and memory grow with input size.

You should now be able to move from code, to a recurrence, to a complexity result, and then explain what that result means.